In [0]:
%python
#---------------------------------------------------------------------------------------------------------------------
'''
Step 1: Create a Managed Table

A managed table does not require a LOCATION.

CREATE TABLE sales_managed (
    id INT,
    amount DOUBLE
);

INSERT INTO sales_managed VALUES
(1,100),
(2,200);

Databricks stores the data in its managed storage area.

Verify:
DESCRIBE DETAIL sales_managed;

You will see a location automatically assigned by Databricks.

Step 2: Create a Shallow Clone of Managed Table

CREATE TABLE sales_managed_shallow
SHALLOW CLONE sales_managed;

What happens?

sales_managed
      |
      +-----> Delta Files
      |
sales_managed_shallow

Both tables point to the same underlying files.

Step 3: Create a Deep Clone of Managed Table
CREATE TABLE sales_managed_deep
DEEP CLONE sales_managed;

Now:

sales_managed
      |
      +-----> Delta Files A

sales_managed_deep
      |
      +-----> Delta Files B

Databricks physically copies the files.

Test Managed Table Clones

Check locations:

DESCRIBE DETAIL sales_managed;

DESCRIBE DETAIL sales_managed_shallow;

DESCRIBE DETAIL sales_managed_deep;

Expected:

Table	Location
sales_managed	Path A
sales_managed_shallow	New table metadata, references files in Path A
sales_managed_deep	Path B
Step 4: Create an External Table

External tables require LOCATION.

Example:

CREATE TABLE sales_external (
    id INT,
    amount DOUBLE
)
LOCATION 'abfss://container@storageacct.dfs.core.windows.net/sales_external';

Insert data:

INSERT INTO sales_external VALUES
(1,100),
(2,200);

Now data lives in your ADLS path.

Step 5: Shallow Clone External Table
CREATE TABLE sales_external_shallow
SHALLOW CLONE sales_external;

Result:

sales_external
        |
        +------> ADLS Files

sales_external_shallow

Both tables reference the same ADLS files.

Step 6: Deep Clone External Table
CREATE TABLE sales_external_deep
DEEP CLONE sales_external;

Databricks copies the Delta files.

Result:

sales_external
        |
        +------> ADLS Files A

sales_external_deep
        |
        +------> ADLS Files B
The Important Part

Many people think:

"External table means deep clone cannot copy data."

That's incorrect for Delta tables.

A deep clone of an external Delta table creates a new copy of the Delta files.

How to Prove It

Run:

DESCRIBE DETAIL sales_external;
DESCRIBE DETAIL sales_external_deep;

Look at the location.

You should see different paths.

Example:

sales_external
abfss://container/data/sales_external

sales_external_deep
abfss://container/data/sales_external_deep

Different locations = separate files.

What Happens If Source Is Dropped?
Managed Table
Shallow Clone
DROP TABLE sales_managed;

Shallow clone may become unusable after source files are cleaned up (for example, by VACUUM).

Deep Clone

Still works.

External Table
Shallow Clone
DROP TABLE sales_external;

Usually still works because dropping the table removes metadata only, not the ADLS files.

However:

DELETE FILES
VACUUM
Manual ADLS deletion

can break the shallow clone because it depends on those files.

Deep Clone

Still works because it owns copied files.

Easy Interview Answer
Source Table Type	Clone Type	Data Copied?	Independent?
Managed	Shallow	No	No
Managed	Deep	Yes	Yes
External	Shallow	No	No
External	Deep	Yes	Yes
'''

In [0]:
USE CATALOG delta_catalog;

In [0]:
USE CATALOG delta_catalog;
CREATE SCHEMA delta_catalog.demo_schema1 
MANAGED LOCATION 'abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/catalog_111/schemaloc';

In [0]:
CREATE TABLE demo_schema1.demo_table1
USING DELTA
LOCATION 'abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/catalog_111/schemaloc/demotable'
AS
SELECT 
  CAST(Customerid AS INT) AS Customerid,
  Surname,
  CAST(Creditscore AS INT) AS Creditscore,
  Geography,
  Gender,
  CAST(Age AS INT) AS Age,
  CAST(Tenure AS INT) AS Tenure,
  CAST(Balance AS DOUBLE) AS Balance,
  CAST(Estimatedsalary AS DOUBLE) AS Etimatedsalary
FROM read_files(
  'abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/bankchurn/bankchurn_1_50.csv',
  format => 'csv',
  header => 'true'
);

In [0]:
CREATE TABLE demo_schema1.shallow_clonetable
SHALLOW CLONE demo_schema1.demo_table1
LOCATION 'abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/catalog_111/schemaloc/demotable_shallow'


In [0]:
select * from demo_schema1.demo_table1

In [0]:
select * from delta_catalog.demo_schema1.shallow_clonetable

In [0]:
CREATE TABLE demo_schema1.deep_clonetable
DEEP CLONE demo_schema1.demo_table1
LOCATION 'abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/catalog_111/schemaloc/demotable_deep'

In [0]:
%python
for i in range (1,5):
     spark.sql("INSERT INTO delta_catalog.demo_schema1.demo_table1 VALUES (1100051,'Barak',444,'France','Male',45,5,84350.07,243835.76)")

In [0]:
select * from delta_catalog.demo_schema1.demo_table1

In [0]:
select * from delta_catalog.demo_schema1.deep_clonetable

In [0]:
select * from demo_schema1.shallow_clonetable

In [0]:
CREATE TABLE demo_schema1.deep_clonetable1
DEEP CLONE demo_schema1.demo_table1 VERSION AS OF 0
LOCATION 'abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/catalog_111/schemaloc/demotable_deep1'

In [0]:
select * from demo_schema1.deep_clonetable1

In [0]:
CREATE TABLE demo_schema1.shallow_clonetable1
SHALLOW CLONE demo_schema1.demo_table1 VERSION AS OF 0
LOCATION 'abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/catalog_111/schemaloc/demotable_shallow1'

In [0]:
select * from demo_schema1.shallow_clonetable1

In [0]:
SHOW CREATE TABLE demo_schema1.shallow_clonetable1


In [0]:

SHOW CREATE TABLE demo_schema1.demo_table1

In [0]:
%python
#---------------------------------------------------------------------------------------------------------------------
'''
Step 1: Create a Managed Table

A managed table does not require a LOCATION.

CREATE TABLE sales_managed (
    id INT,
    amount DOUBLE
);

INSERT INTO sales_managed VALUES
(1,100),
(2,200);

Databricks stores the data in its managed storage area.

Verify:

DESCRIBE DETAIL sales_managed;

You will see a location automatically assigned by Databricks.

Step 2: Create a Shallow Clone of Managed Table
CREATE TABLE sales_managed_shallow
SHALLOW CLONE sales_managed;

What happens?

sales_managed
      |
      +-----> Delta Files
      |
sales_managed_shallow

Both tables point to the same underlying files.

Step 3: Create a Deep Clone of Managed Table
CREATE TABLE sales_managed_deep
DEEP CLONE sales_managed;

Now:

sales_managed
      |
      +-----> Delta Files A

sales_managed_deep
      |
      +-----> Delta Files B

Databricks physically copies the files.

Test Managed Table Clones

Check locations:

DESCRIBE DETAIL sales_managed;

DESCRIBE DETAIL sales_managed_shallow;

DESCRIBE DETAIL sales_managed_deep;

Expected:

Table	Location
sales_managed	Path A
sales_managed_shallow	New table metadata, references files in Path A
sales_managed_deep	Path B
Step 4: Create an External Table

External tables require LOCATION.

Example:

CREATE TABLE sales_external (
    id INT,
    amount DOUBLE
)
LOCATION 'abfss://container@storageacct.dfs.core.windows.net/sales_external';

Insert data:

INSERT INTO sales_external VALUES
(1,100),
(2,200);

Now data lives in your ADLS path.

Step 5: Shallow Clone External Table
CREATE TABLE sales_external_shallow
SHALLOW CLONE sales_external;

Result:

sales_external
        |
        +------> ADLS Files

sales_external_shallow

Both tables reference the same ADLS files.

Step 6: Deep Clone External Table
CREATE TABLE sales_external_deep
DEEP CLONE sales_external;

Databricks copies the Delta files.

Result:

sales_external
        |
        +------> ADLS Files A

sales_external_deep
        |
        +------> ADLS Files B
The Important Part

Many people think:

"External table means deep clone cannot copy data."

That's incorrect for Delta tables.

A deep clone of an external Delta table creates a new copy of the Delta files.

How to Prove It

Run:

DESCRIBE DETAIL sales_external;
DESCRIBE DETAIL sales_external_deep;

Look at the location.

You should see different paths.

Example:

sales_external
abfss://container/data/sales_external

sales_external_deep
abfss://container/data/sales_external_deep

Different locations = separate files.

What Happens If Source Is Dropped?
Managed Table
Shallow Clone
DROP TABLE sales_managed;

Shallow clone may become unusable after source files are cleaned up (for example, by VACUUM).

Deep Clone

Still works.

External Table
Shallow Clone
DROP TABLE sales_external;

Usually still works because dropping the table removes metadata only, not the ADLS files.

However:

DELETE FILES
VACUUM
Manual ADLS deletion

can break the shallow clone because it depends on those files.

Deep Clone

Still works because it owns copied files.

Easy Interview Answer
Source Table Type	Clone Type	Data Copied?	Independent?
Managed	Shallow	No	No
Managed	Deep	Yes	Yes
External	Shallow	No	No
External	Deep	Yes	Yes
'''

In [0]:
SHOW CREATE TABLE demo_schema1.deep_clonetable1

In [0]:
CREATE TABLE demo_schema1.ext_sales
USING DELTA
LOCATION "abfss://testdatabricksdemo@datbricksstrgaccextdl.dfs.core.windows.net/catalog_111/schemaloc/extsales"

In [0]:
CREATE TABLE demo_schema1.ext_sales_shallow
SHALLOW CLONE demo_schema1.ext_sales;